# 07 · Sensor streams and uncertainty

**Question:** Can an uncertain model track a changing urban heat signal from noisy observations?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
All city geometry, residents, sensor observations and parameters are **synthetic educational fixtures**.
The longitude/latitude anchor is near Gaborone, but these are not mapped Gaborone assets or a validated city twin.

**Astra experiment:** Ask Astra to diagnose an outlier and missing readings and justify whether uncertainty should increase.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## A one-dimensional Kalman filter
One representative location has a random-walk latent signal. A sensor occasionally drops readings and has one large outlier.
The filter rejects readings beyond four predicted standard deviations. This is a teaching model, not a citywide spatial filter.


In [ ]:
rng=np.random.default_rng(42); steps=80; process_var=.12; sensor_var=1.5
truth=30+np.cumsum(rng.normal(0,np.sqrt(process_var),steps))
observed=truth+rng.normal(0,np.sqrt(sensor_var),steps)
observed[20:28]=np.nan; observed[45]+=15
estimate=[]; variance=[]; rejected=[]; mean=30.; var=4.
for t,obs in enumerate(observed):
    var+=process_var
    if np.isfinite(obs):
        innovation=obs-mean
        if abs(innovation)<=4*np.sqrt(var+sensor_var):
            gain=var/(var+sensor_var); mean+=gain*innovation; var*=(1-gain)
        else: rejected.append(t)
    estimate.append(mean); variance.append(var)
estimate=np.array(estimate); variance=np.array(variance)
assert (variance>0).all() and 45 in rejected
assert variance[27]>variance[19]
rmse=float(np.sqrt(np.mean((estimate-truth)**2)))
print('Filter RMSE:',rmse,'| rejected indices:',rejected)


In [ ]:
t=np.arange(steps); fig,ax=plt.subplots()
ax.fill_between(t,estimate-1.96*np.sqrt(variance),estimate+1.96*np.sqrt(variance),alpha=.2,label='Approximate 95% model interval')
ax.plot(t,truth,label='Synthetic truth'); ax.plot(t,estimate,label='Estimate')
ax.scatter(t,observed,s=12,c='#e45756',label='Sensor')
ax.set(xlabel='Time step',ylabel='Synthetic degrees C',title='Assimilation with missing and rejected readings'); ax.legend(); plt.show()
export_json('07_assimilation.json',dict(synthetic=True,rmse=rmse,rejected=rejected,process_variance=process_var,sensor_variance=sensor_var))


## Teaching lab: Diagnose uncertainty
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 07_assimilation.json and the sensor plot. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Explain the missing interval and rejected outlier. Distinguish a model uncertainty band from empirically verified interval coverage.

**Copyable Codex task:**
> Run the filter on multiple seeded streams and measure empirical interval coverage against synthetic truth. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** Missing readings increase prediction variance; changing sensor noise changes gain; report coverage without forcing it to 95%.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extend with Codex or Astra
Add timestamp validation, stale-data indicators and multiple locations. Evaluate interval coverage on held-out runs and test sensitivity to mis-specified noise.

Keep the baseline seed and tests fixed while changing one assumption. Record the model name,
date, exact prompt, response and error metrics. Generated code must pass the same checks.
Download files in `exports/` and your edited notebook; browser storage does not commit changes to GitHub.
See [the project guide](https://github.com/jltobias/JupyterLite-GPT-6-Astra-Geospatial/blob/main/docs/GUIDE.md)
for capabilities, evidence, sources and the route to real data.
